# Projeto de Análise de Mercado Automotivo (EUA)
Este notebook contém a Análise Exploratória de Dados (EDA) para entender os fatores que influenciam o preço e o tempo de venda de veículos usados, tendo como base uma tabela pré-existente de mercado. 

**Objetivos:**
- Limpeza e tratamento de dados ausentes.
- Análise de correlação entre quilometragem e preço.
- Estudo do impacto da tração 4x4 no valor de revenda.

In [39]:
# Comandos utilizados para corrigir erros de instalação de pacotes no ambiente virtual do Jupyter Notebook.
#      
#  !pip install pandas plotly import sys#
#import sys
# python = sys.executable
# !$python -m pip install pandas plotly nbformat 
# %pip install pandas plotly nbformat 
# !pip install statsmodels



In [40]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# Define o caminho para a pasta onde o notebook está
current_dir = Path.cwd()

# Lista de nomes possíveis para o arquivo (na pasta raiz, um nível acima)
# Tentamos encontrar na pasta pai (..)
candidate_files = [
    current_dir.parent / "vehicles_us.csv",
    current_dir.parent / "vehicles.csv",
    current_dir.parent / "vehicles_us.csv"
]

# Procura qual deles existe
data_path = next((p for p in candidate_files if p.exists()), None)

if data_path:
    df = pd.read_csv(data_path)
    print(f"Sucesso! Arquivo encontrado em: {data_path}")
    print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")
    display(df.head())
else:
    print("ERRO: O arquivo CSV não foi encontrado na pasta raiz.")
    print(f"Arquivos na pasta raiz: {[f.name for f in current_dir.parent.iterdir()]}")

Sucesso! Arquivo encontrado em: c:\Users\rogerio.barberi\OneDrive - Dracones Consultoria em TI\Pessoal\Triple Ten\Cinconotebook\vehicles_env\vehicles_us.csv
O dataset possui 51525 linhas e 13 colunas.


,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [41]:
# Verificando valores ausentes e tipos de dados
print(df.info())
print("\nValores nulos por coluna:")
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 5.1 MB
None

Valores nulos por coluna:
price               0
model_year       3619
model               0
condition           0
cylinders        5260
fuel                0
odometer         7892
transmission 

### 🛠️ Tratamento de Dados Ausentes
Para evitar a perda de aproximadamente 20% do dataset, aplicamos a **Imputação por Mediana de Grupo**:

* **Ano do Modelo e Cilindros:** Preenchidos pela mediana de cada `model` específico.
* **Quilometragem (odometer):** Preenchida pela mediana correspondente ao `model_year`, respeitando a correlação entre idade e uso.

> **Nota de Integridade:** Esta técnica preserva as características específicas de cada categoria de veículo, evitando as distorções que ocorreriam caso usássemos uma média global.

In [42]:
# 1. Imputação inteligente
df['model_year'] = df['model_year'].fillna(df.groupby('model')['model_year'].transform('median'))
df['cylinders'] = df['cylinders'].fillna(df.groupby('model')['cylinders'].transform('median'))
df['odometer'] = df['odometer'].fillna(df.groupby('model_year')['odometer'].transform('median'))

# 2. Padronização de booleanos
df['is_4wd'] = df['is_4wd'].fillna(0)

# 3. Criação da coluna Brand (Marca)
df['brand'] = df['model'].apply(lambda x: str(x).split()[0])

# Removendo nulos restantes que não puderam ser imputados
df.dropna(subset=['price', 'model_year', 'odometer'], inplace=True)

print("Nulos restantes:", df.isna().sum().sum())

Nulos restantes: 9267


In [43]:
# Testando o histograma que irá para o App
fig = px.histogram(df, x="price", title="Distribuição de Preços - Teste EDA")
fig.show()

# Testando o gráfico de dispersão
fig_scatter = px.scatter(df, x="odometer", y="price", opacity=0.3, title="Preço vs Quilometragem")
fig_scatter.show()


# 1. Análise de Liquidez: O que vende mais rápido?
Nesta seção, investigamos quais tipos de veículos possuem o maior giro de estoque. 
A métrica principal é `days_listed` (dias em anúncio).
**Hipótese:** Veículos utilitários (Trucks/SUVs) possuem menor tempo de anúncio devido à alta demanda.

In [44]:
# Agrupando por tipo e calculando a média de dias listados
liquidity = df.groupby('type')['days_listed'].mean().sort_values().reset_index()

fig_liq = px.bar(liquidity, 
                 x='days_listed', 
                 y='type', 
                 orientation='h',
                 title='Média de Dias em Anúncio por Categoria',
                 labels={'days_listed':'Média de Dias', 'type':'Categoria'},
                 color='days_listed',
                 color_continuous_scale='Viridis')
fig_liq.show()

Ao analisar a média de dias que um veículo permanece anunciado, observamos padrões distintos dependendo da faixa de preço:

1. **Mercado Geral (Até $150.000):** As categorias **Conversível**, **Pickup** e **Coupe** lideram a velocidade de venda. Isso sugere um mercado de nicho e de utilitários pesados muito aquecido.
2. **Mercado de Volume (Até $30.000):** Quando filtramos para veículos mais acessíveis, a categoria **Truck** ganha relevância, superando os modelos Coupe. Isso indica que, na faixa de preço de entrada/média, os veículos de trabalho (Trucks/Pickups) são os ativos com maior liquidez.
3. **Gargalo de Venda:** Em todos os cenários, a categoria **Bus** (Ônibus) é a que permanece mais tempo em anúncio (média acima de 43 dias), o que é esperado devido ao público muito específico e ao alto valor agregado/custo de manutenção desses veículos.

**Conclusão:** Para um revendedor, investir em Pickups e Trucks na faixa de até $30.000 garante o retorno de capital mais rápido (menor tempo de pátio).

# 2. Fatores de Valorização: O "Prêmio" do 4x4
Nesta seção, analisamos se a presença de tração integral (`is_4wd`) influencia significativamente o preço de revenda dos veículos.

In [45]:
# Criando o gráfico de Boxplot para comparar 4x4 vs Tração Simples
fig_4wd = px.box(df, 
                 x='is_4wd', 
                 y='price', 
                 color='is_4wd',
                 title='Impacto da Tração 4x4 no Preço de Venda',
                 labels={'is_4wd': 'Possui 4x4', 'price': 'Preço ($)'},
                 category_orders={"is_4wd": [0, 1]})

# Limitando o eixo Y para 50.000 para ignorar outliers e ver a caixa principal
fig_4wd.update_yaxes(range=[0, 50000])
fig_4wd.show()

### Análise do Prêmio de Valor: Tração 4x4

A análise visual dos Boxplots revela um impacto substancial da tração integral no preço de revenda:

1. **Valor de Mercado:** A mediana de preço para veículos com tração 4x4 é significativamente superior à de veículos com tração simples (aproximadamente **$13k vs $7k**).
2. **Distribuição de Preço:** Observa-se que o limite superior do terceiro quartil (topo da caixa) para veículos 4x2 mal atinge a mediana dos veículos 4x4. Isso indica que a tração integral coloca o veículo em um patamar de preço superior de forma consistente.
3. **Público Alvo:** Este "prêmio" no preço sugere que o comprador de usados valoriza a capacidade off-road ou a segurança em climas adversos, estando disposto a pagar quase o dobro por essa funcionalidade.

**Insight de Negócio:** Em inventários de revenda, priorizar a aquisição de modelos 4x4 pode garantir margens de lucro maiores, dado que o teto de preço é consideravelmente mais alto.

# 3. Estética e Valor: A Influência das Cores
Nesta seção, exploramos se existe uma correlação entre a cor externa do veículo (`paint_color`) e o preço médio anunciado. 
**Hipótese:** Cores sóbrias (Branco, Preto, Prata) retêm melhor o valor por terem maior facilidade de revenda.

In [46]:
# Calculando a mediana de preço por cor para evitar distorção de outliers
color_price = df.groupby('paint_color')['price'].median().sort_values(ascending=False).reset_index()

fig_color = px.bar(color_price, 
                   x='paint_color', 
                   y='price', 
                   color='price',
                   title='Mediana de Preço por Cor do Veículo',
                   labels={'paint_color': 'Cor', 'price': 'Preço Mediano ($)'},
                   color_continuous_scale='Reds')
fig_color.show()

### Análise de Influência Estética: Cor vs. Valor Mediano

A análise de preços por cor revela um padrão contraintuitivo que desafia o senso comum do mercado:

1. **Cores de "Status" e Nicho:** As cores **Amarelo (Yellow)** e **Laranja (Orange)** apresentam as maiores medianas de preço (acima de $15k). Isso não ocorre necessariamente porque a tinta é mais cara, mas porque essas cores são frequentemente associadas a modelos específicos de alto valor, como carros esportivos, edições limitadas de SUVs e veículos de lazer.
2. **Cores Comerciais:** As cores mais comuns (Branco, Preto, Cinza e Prata) ocupam o pelotão intermediário, com medianas entre **$8k e $12k**. Elas representam a estabilidade do mercado de volume.
3. **Depreciação por Excentricidade:** No extremo inferior, cores como **Verde (Green)** e **Roxo (Purple)** apresentam as menores medianas. Isso sugere que cores com menor aceitação popular podem sofrer uma desvalorização mais acentuada devido à baixa liquidez no mercado de usados.

**Insight de Negócio:** Para uma revendedora, adquirir veículos em cores "comerciais" reduz o risco, mas modelos em cores vibrantes podem representar oportunidades de ticket médio mais elevado, desde que o modelo do veículo suporte essa estética.

# 4. Ciclo de Vida: Depreciação por Quilometragem e Condição
Nesta análise final, observamos a taxa de desvalorização dos veículos em relação às milhas rodadas (`odometer`), estratificando pelo estado de conservação (`condition`).

In [47]:
# Gráfico de Dispersão com Linha de Tendência (Trendline)
fig_depre = px.scatter(df, 
                       x="odometer", 
                       y="price", 
                       color="condition",
                       opacity=0.3, # Lembra do erro? Usamos opacity aqui no Notebook também.
                       trendline="lowess", # Adiciona uma linha de tendência suave
                       title="Relação Preço x Quilometragem por Condição",
                       labels={'odometer': 'Milhas Rodadas', 'price': 'Preço ($)'})

fig_depre.update_layout(xaxis_range=[0, 300000], yaxis_range=[0, 80000])
fig_depre.show()

### Análise de Depreciação: Preço vs. Odômetro por Condição

Este gráfico de dispersão com linhas de tendência (Lowess) fornece os insights mais profundos sobre o ciclo de vida financeiro dos veículos:

1. **A Queda Livre do "Zero KM":** A linha azul clara (**new**) apresenta a inclinação mais acentuada de todas. Isso comprova visualmente que a maior perda de valor ocorre nos primeiros milhares de milhas rodadas. 
2. **Convergência de Valor por Desgaste:** Observe que, conforme os veículos ultrapassam a marca das **150.000 milhas**, as linhas de tendência para condições "Excellent", "Good" e "Fair" começam a convergir. Isso indica que, para o mercado de alta quilometragem, o estado estético perde relevância para o valor funcional residual.
3. **Resiliência do Estado de Conservação:** Entre 50k e 120k milhas, existe um "gap" (espaço) claro entre as linhas. Um veículo em condição **Excellent** (roxo) mantém um prêmio de preço constante sobre um veículo **Good** (azul escuro) nessa faixa, justificando investimentos em manutenção estética para revenda.
4. **O "Piso" de Sucata:** Mesmo após as 250.000 milhas, os preços tendem a se estabilizar em um patamar mínimo (piso), sugerindo que o veículo mantém um valor base enquanto for capaz de operar, independentemente da idade.

**Insight Estratégico:** Para maximizar o retorno sobre o capital, o momento ideal de revenda é antes das **100.000 milhas**, ponto onde a curva de depreciação começa a se achatar e o diferencial de "condição" começa a desaparecer.

# 5. Conclusão Geral do Projeto

Após a análise exploratória dos dados de anúncios de veículos, chegamos às seguintes conclusões estratégicas:

* **Liquidez de Mercado:** O mercado de usados é movido pela utilidade. Veículos do tipo **Truck** e **Pickup** na faixa de até **$30.000** apresentam o menor tempo médio de anúncio, sendo os ativos de maior giro para uma revendedora.
* **Fatores de Valorização:** A presença de **tração 4x4** atua como um divisor de águas no preço, chegando a dobrar a mediana de valor em comparação com modelos de tração simples. 
* **Impacto da Estética:** Cores vibrantes (Amarelo/Laranja) estão associadas a nichos de maior valor, enquanto o Roxo e o Verde apresentam maior desvalorização. Cores neutras (Branco/Preto) mantêm a estabilidade do mercado de massa.
* **Ciclo de Depreciação:** A desvalorização é mais agressiva nas primeiras **50.000 milhas**. Após as **150.000 milhas**, o estado de conservação do veículo torna-se secundário ao preço, que tende a se estabilizar em um valor residual funcional.

**Recomendação:** Para otimizar o estoque, deve-se priorizar a aquisição de Pickups/Trucks 4x4, preferencialmente com menos de 100.000 milhas, onde o valor de revenda ainda protege a margem de lucro contra a depreciação acelerada.